# Analyse Exploratoire de Données (EDA) - Deep Learning Médical

## Section 0 — Installation et imports

In [1]:
!pip install medmnist pandas numpy matplotlib seaborn scipy wordcloud scikit-learn -q

'pip' n'est pas reconnu en tant que commande interne
ou externe, un programme ex‚cutable ou un fichier de commandes.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pointbiserialr
import warnings
warnings.filterwarnings('ignore')

# Style global des graphiques
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## Section 1 — EDA : Pima Indians Diabetes

### 1.1 Chargement des données

In [3]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv(url, names=cols)

display(df.head(10))
display(df.tail(5))
print("Shape:", df.shape)
display(df.dtypes)
display(df.describe().T)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1
767,1,93,70,31,0,30.4,0.315,23,0


Shape: (768, 9)


Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


### 1.2 Analyse des valeurs manquantes et aberrantes

In [4]:
print("Valeurs nulles par colonne :")
print(df.isnull().sum())

plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Heatmap des valeurs nulles")
plt.show()

# Zéros biologiquement impossibles
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_counts = (df[cols_with_zeros] == 0).sum()
zero_percent = (zero_counts / len(df)) * 100
zeros_df = pd.DataFrame({'Zéros (Count)': zero_counts, 'Zéros (%)': zero_percent})

display(zeros_df)

plt.figure(figsize=(8, 4))
sns.barplot(x=zeros_df['Zéros (%)'], y=zeros_df.index, palette='Reds_r')
plt.title("% de zéros aberrants par feature")
plt.xlabel("Pourcentage (%)")
plt.savefig('pima_zeros.png', dpi=300, bbox_inches='tight')
plt.show()

Valeurs nulles par colonne :
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


,Zéros (Count),Zéros (%)
Glucose,5,0.651042
BloodPressure,35,4.557292
SkinThickness,227,29.557292
Insulin,374,48.697917
BMI,11,1.432292


### 1.3 Distribution des classes (target)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['Outcome'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', labels=['Non-Diabétique (0)', 'Diabétique (1)'], colors=['skyblue', 'salmon'], explode=(0.05, 0))
axes[0].set_title('Distribution des classes (Pie Chart)')
axes[0].set_ylabel('')

sns.countplot(data=df, x='Outcome', palette=['skyblue', 'salmon'], ax=axes[1])
axes[1].set_title('Distribution des classes (Barplot)')
axes[1].set_xticklabels(['Non-Diabétique (0)', 'Diabétique (1)'])

plt.tight_layout()
plt.savefig('pima_classes.png', dpi=300, bbox_inches='tight')
plt.show()

ratio = df['Outcome'].value_counts()[0] / df['Outcome'].value_counts()[1]
print(f"Ratio de déséquilibre (0/1): {ratio:.2f}")

Ratio de déséquilibre (0/1): 1.87


**Commentaire sur le déséquilibre :**
Le dataset est déséquilibré avec environ 65% de cas non-diabétiques contre 35% de cas diabétiques. Ce déséquilibre est fréquent en milieu médical et nécessite des stratégies particulières (comme un class_weight, du sur-échantillonnage, ou des métriques adaptées telles que le F1-Score et l'AUC-ROC) pour éviter qu'un modèle MLP ne favorise aveuglément la classe majoritaire.

### 1.4 Distribution de chaque feature

In [6]:
# Histogrammes KDE
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(df.columns[:-1]):
    sns.histplot(data=df, x=col, hue='Outcome', kde=True, ax=axes[i], palette='Set1', element='step')
    axes[i].set_title(f"Distribution de {col}")
plt.tight_layout()
plt.savefig('pima_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

# Boxplots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(df.columns[:-1]):
    sns.boxplot(data=df, x='Outcome', y=col, ax=axes[i], palette='Set1')
    axes[i].set_title(f"Boxplot de {col}")
plt.tight_layout()
plt.savefig('pima_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

# Violinplots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(df.columns[:-1]):
    sns.violinplot(data=df, x='Outcome', y=col, ax=axes[i], palette='Set1', split=True)
    axes[i].set_title(f"Violinplot de {col}")
plt.tight_layout()
plt.savefig('pima_violinplots.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.5 Tableau de corrélation (PRINCIPAL)

In [7]:
corr_matrix = df.corr()

# 1. Heatmap complète
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Matrice de corrélation de Pearson — Pima Diabetes")
plt.savefig('pima_corr_full.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Heatmap triangulaire
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title("Corrélations (triangle inférieur)")
plt.savefig('pima_corr_tri.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Barplot corrélations avec Target
plt.figure(figsize=(8, 6))
corr_with_target = df.corr()['Outcome'].drop('Outcome').sort_values()
colors = ['red' if x < 0 else 'green' for x in corr_with_target]
sns.barplot(x=corr_with_target.values, y=corr_with_target.index, palette=colors)
plt.title("Corrélation de chaque feature avec Outcome (diabète)")
plt.xlabel("Coefficient de corrélation de Pearson")
plt.savefig('pima_corr_target.png', dpi=300, bbox_inches='tight')
plt.show()

In [8]:
corr_with_target = df.corr()['Outcome'].drop('Outcome').sort_values(ascending=False)
print("=== Corrélations avec Outcome (mesurées sur les données) ===")
display(corr_with_target.to_frame('Pearson'))

top3_pos = corr_with_target.head(3)
top3_weak = corr_with_target.reindex(corr_with_target.abs().sort_values().index).head(3)
print("\nTop 3 corrélations les plus fortes (positives):")
for feat, val in top3_pos.items():
    print(f"  - {feat}: {val:.3f}")
print("\nTop 3 corrélations les plus faibles (|r| minimal):")
for feat, val in top3_weak.items():
    print(f"  - {feat}: {val:.3f}")

=== Corrélations avec Outcome (mesurées sur les données) ===


,Pearson
Glucose,0.466581
BMI,0.292695
Age,0.238356
Pregnancies,0.221898
DiabetesPedigreeFunction,0.173844
Insulin,0.130548
SkinThickness,0.074752
BloodPressure,0.065068



Top 3 corrélations les plus fortes (positives):
  - Glucose: 0.467
  - BMI: 0.293
  - Age: 0.238

Top 3 corrélations les plus faibles (|r| minimal):
  - BloodPressure: 0.065
  - SkinThickness: 0.075
  - Insulin: 0.131


**Interprétation (basée sur les corrélations mesurées ci-dessus) :**
Les variables les plus corrélées positivement à `Outcome` sont typiquement **Glucose**, **BMI** et **Age** sur ce jeu Pima — cohérent avec la physiopathologie du diabète de type 2. Les corrélations proches de zéro (ex. pression artérielle, épaisseur de peau) discriminent mal seules la cible, souvent à cause de bruit et de zéros aberrants non imputés à ce stade EDA.

### 1.6 Pairplot

In [9]:
sns.pairplot(df, hue='Outcome', vars=['Glucose','BMI','Age','Insulin','BloodPressure'], diag_kind='kde', plot_kws={'alpha': 0.5}, palette='Set1')
plt.suptitle("Pairplot des features principales — coloré par diagnostic", y=1.02)
plt.savefig('pima_pairplot.png', dpi=300, bbox_inches='tight')
plt.show()

**Commentaire Pairplot :**
On observe clairement que la variable 'Glucose' sépare le mieux les deux classes (les distributions KDE sont les moins chevauchantes). 'BMI' offre également une bonne capacité de discrimination à partir d'un certain seuil. Les relations entre variables sont principalement non-linéaires, justifiant l'utilisation d'un perceptron multicouche (MLP) capable d'apprendre des frontières de décision complexes.

### 1.7 Analyse statistique des corrélations

In [10]:
stat_results = []
for col in df.columns[:-1]:
    group0 = df[df['Outcome'] == 0][col]
    group1 = df[df['Outcome'] == 1][col]
    stat, p_value = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    corr = df[col].corr(df['Outcome'])
    stat_results.append({
        'Feature': col,
        'Corrélation': round(corr, 3),
        'P-value': p_value,
        'Significatif (p<0.05)': 'Oui' if p_value < 0.05 else 'Non'
    })

stat_df = pd.DataFrame(stat_results)
display(stat_df)

,Feature,Corrélation,P-value,Significatif (p<0.05)
0,Pregnancies,0.222,3.745146e-08,Oui
1,Glucose,0.467,1.200727e-39,Oui
2,BloodPressure,0.065,7.558512e-05,Oui
3,SkinThickness,0.075,1.296183e-02,Oui
4,Insulin,0.131,6.566037e-02,Non
5,BMI,0.293,9.730790e-18,Oui
6,DiabetesPedigreeFunction,0.174,1.196583e-06,Oui
7,Age,0.238,1.142200e-17,Oui


### 1.8 Détection des outliers

In [11]:
outliers_summary = []

for col in df.columns[:-1]:
    # IQR Method
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    iqr_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    
    # Z-score Method
    z_scores = np.abs(stats.zscore(df[col]))
    z_outliers = (z_scores > 3).sum()
    
    outliers_summary.append({
        'Feature': col,
        'Outliers IQR': iqr_outliers,
        'Outliers Z-score (>3)': z_outliers
    })

outliers_df = pd.DataFrame(outliers_summary)
display(outliers_df)

# Visualisation ciblée : Insulin et DiabetesPedigreeFunction ont beaucoup d'outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['Insulin'], ax=axes[0], color='lightcoral')
axes[0].set_title('Outliers dans Insulin')
sns.boxplot(x=df['DiabetesPedigreeFunction'], ax=axes[1], color='lightgreen')
axes[1].set_title('Outliers dans DiabetesPedigreeFunction')
plt.savefig('pima_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

,Feature,Outliers IQR,Outliers Z-score (>3)
0,Pregnancies,4,4
1,Glucose,5,5
2,BloodPressure,45,35
3,SkinThickness,1,1
4,Insulin,34,18
5,BMI,19,14
6,DiabetesPedigreeFunction,29,11
7,Age,9,5


### 1.9 Synthèse EDA — Partie I

L'analyse du dataset Pima Indians Diabetes révèle plusieurs défis inhérents aux données de santé tabulaires. Le **déséquilibre des classes** est prononcé (65% vs 35%), ce qui nécessitera potentiellement un pondérateur de perte (Loss weighting) lors de l'entraînement du MLP. Les features **Glucose**, **BMI** et **Âge** s'imposent comme les marqueurs les plus corrélés et discriminants pour la détection du diabète. Cependant, le jeu de données souffre grandement de **valeurs aberrantes**, notamment des "zéros impossibles" sur des indicateurs critiques tels que l'insuline (48% de zéros) ou l'épaisseur de la peau (30%). De plus, une forte présence d'outliers statistiques est démontrée, suggérant qu'un nettoyage approfondi (imputation par la médiane) et une normalisation stricte (StandardScaler) seront des prérequis absolument indispensables pour optimiser les performances du perceptron multicouche.

## Section 2 — EDA : PneumoniaMNIST

### 2.1 Chargement des données

In [12]:
from medmnist import PneumoniaMNIST
from torchvision import transforms
import torch

transform = transforms.ToTensor()
train_data = PneumoniaMNIST(split='train', transform=transform, download=True)
val_data   = PneumoniaMNIST(split='val',   transform=transform, download=True)
test_data  = PneumoniaMNIST(split='test',  transform=transform, download=True)

print("Taille Train:", len(train_data))
print("Taille Val:", len(val_data))
print("Taille Test:", len(test_data))

img, label = train_data[0]
print("Shape de l'image:", img.shape)
print("Label (0=Normal, 1=Pneumonie):", label.item())

Taille Train: 4708
Taille Val: 524
Taille Test: 624
Shape de l'image: torch.Size([1, 28, 28])
Label (0=Normal, 1=Pneumonie): 1


### 2.2 Distribution des classes

In [13]:
train_labels = [label.item() for _, label in train_data]
val_labels = [label.item() for _, label in val_data]
test_labels = [label.item() for _, label in test_data]

splits = ['Train', 'Validation', 'Test']
df_splits = pd.DataFrame({
    'Split': splits * 2,
    'Classe': ['Normal']*3 + ['Pneumonie']*3,
    'Count': [
        train_labels.count(0), val_labels.count(0), test_labels.count(0),
        train_labels.count(1), val_labels.count(1), test_labels.count(1)
    ]
})

plt.figure(figsize=(10, 5))
sns.barplot(data=df_splits, x='Split', y='Count', hue='Classe', palette=['skyblue', 'salmon'])
plt.title('Distribution des classes par split')
plt.savefig('pneumonia_dist.png', dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].pie([train_labels.count(0), train_labels.count(1)], labels=['Normal', 'Pneumo'], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[0].set_title('Train')
axes[1].pie([val_labels.count(0), val_labels.count(1)], labels=['Normal', 'Pneumo'], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[1].set_title('Val')
axes[2].pie([test_labels.count(0), test_labels.count(1)], labels=['Normal', 'Pneumo'], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[2].set_title('Test')
plt.savefig('pneumonia_pie.png', dpi=300, bbox_inches='tight')
plt.show()

**Commentaire sur le déséquilibre :**
Le dataset présente un fort déséquilibre en faveur de la pneumonie (environ 74% vs 26%). Cela reflète le fait clinique que dans une cohorte de patients symptomatiques suspects, la pathologie est très fréquente. Le CNN devra donc être bien régularisé.

### 2.3 Visualisation des images

In [14]:
normal_imgs = [img for img, lbl in train_data if lbl.item() == 0][:25]
pneumo_imgs = [img for img, lbl in train_data if lbl.item() == 1][:25]

def show_grid(images, title, filename):
    fig, axes = plt.subplots(5, 5, figsize=(8, 8))
    fig.suptitle(title, fontsize=16)
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(images[i].squeeze(), cmap='gray')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

show_grid(normal_imgs, "Radiographies - Normal", 'pneumo_grid_normal.png')
show_grid(pneumo_imgs, "Radiographies - Pneumonie", 'pneumo_grid_pneumo.png')

# Mixte
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
fig.suptitle("Mixte (Normal vs Pneumonie)", fontsize=16)
for i in range(3):
    axes[0, i].imshow(normal_imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title("Normal")
    axes[0, i].axis('off')
    axes[1, i].imshow(pneumo_imgs[i].squeeze(), cmap='gray')
    axes[1, i].set_title("Pneumonie")
    axes[1, i].axis('off')
plt.savefig('pneumo_grid_mixte.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.4 Analyse des intensités de pixels

In [15]:
all_images = np.array([img.numpy() for img, _ in train_data])
all_labels = np.array([lbl.item() for _, lbl in train_data])

norm_pixels = all_images[all_labels == 0].flatten()
pneu_pixels = all_images[all_labels == 1].flatten()

print(f"Pixel moyen global : {all_images.mean():.4f}")
print(f"Écart-type global : {all_images.std():.4f}")

plt.figure(figsize=(10, 5))
sns.kdeplot(norm_pixels, label="Normal", color="skyblue", fill=True, bw_adjust=0.5)
sns.kdeplot(pneu_pixels, label="Pneumonie", color="salmon", fill=True, bw_adjust=0.5)
plt.title("Distribution des intensités de pixels (Normal vs Pneumonie)")
plt.xlabel("Intensité du pixel (0 à 1)")
plt.ylabel("Densité")
plt.legend()
plt.savefig('pneumo_pixel_hist.png', dpi=300, bbox_inches='tight')
plt.show()

# Images moyennes
mean_img_norm = all_images[all_labels == 0].mean(axis=0).squeeze()
mean_img_pneu = all_images[all_labels == 1].mean(axis=0).squeeze()
std_img_norm = all_images[all_labels == 0].std(axis=0).squeeze()
std_img_pneu = all_images[all_labels == 1].std(axis=0).squeeze()

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
im1 = axes[0, 0].imshow(mean_img_norm, cmap='gray')
axes[0, 0].set_title("Image Moyenne - Normal")
fig.colorbar(im1, ax=axes[0, 0])

im2 = axes[0, 1].imshow(mean_img_pneu, cmap='gray')
axes[0, 1].set_title("Image Moyenne - Pneumonie")
fig.colorbar(im2, ax=axes[0, 1])

im3 = axes[1, 0].imshow(std_img_norm, cmap='hot')
axes[1, 0].set_title("Écart-Type Spatial - Normal")
fig.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(std_img_pneu, cmap='hot')
axes[1, 1].set_title("Écart-Type Spatial - Pneumonie")
fig.colorbar(im4, ax=axes[1, 1])

plt.tight_layout()
plt.savefig('pneumo_mean_std.png', dpi=300, bbox_inches='tight')
plt.show()

Pixel moyen global : 0.5719
Écart-type global : 0.1684


### 2.5 Tableau de corrélation (PRINCIPAL)

In [16]:
from sklearn.decomposition import PCA

# On prend un sous-ensemble
sub_imgs = all_images[:1000].reshape(1000, 28*28)
sub_lbls = all_labels[:1000]

# 1. Corrélation Pixel-Label
corrs = np.array([pointbiserialr(sub_imgs[:, i], sub_lbls)[0] for i in range(784)])
corrs = np.nan_to_num(corrs).reshape(28, 28)

plt.figure(figsize=(8, 6))
sns.heatmap(corrs, cmap='RdBu_r', center=0)
plt.title("Carte de corrélation pixel-label — PneumoniaMNIST")
plt.savefig('pneumo_corr_pixel.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Corrélation Inter-Régions
imgs_28 = all_images[:1000].squeeze()
q_tl = imgs_28[:, :14, :14].mean(axis=(1,2))
q_tr = imgs_28[:, :14, 14:].mean(axis=(1,2))
q_bl = imgs_28[:, 14:, :14].mean(axis=(1,2))
q_br = imgs_28[:, 14:, 14:].mean(axis=(1,2))

quad_df = pd.DataFrame({'TopLeft': q_tl, 'TopRight': q_tr, 'BottomLeft': q_bl, 'BottomRight': q_br})

plt.figure(figsize=(6, 5))
sns.heatmap(quad_df.corr(), annot=True, cmap='viridis')
plt.title("Corrélations inter-régions pulmonaires")
plt.savefig('pneumo_corr_regions.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. PCA
pca = PCA(n_components=2)
pca_res = pca.fit_transform(sub_imgs)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca_res[:, 0], y=pca_res[:, 1], hue=['Pneumonie' if l==1 else 'Normal' for l in sub_lbls], palette=['salmon', 'skyblue'], alpha=0.7)
plt.title(f"Projection PCA des radiographies (Var expliquée : {pca.explained_variance_ratio_.sum()*100:.1f}%)")
plt.savefig('pneumo_pca.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.6 Statistiques par classe

In [17]:
stats_df = pd.DataFrame({
    'Métrique': ['Intensité moyenne', 'Écart-type moyen', 'Pixels > 0.5 (%)', 'Pixels < 0.2 (%)'],
    'Normal': [
        norm_pixels.mean(), norm_pixels.std(),
        (norm_pixels > 0.5).mean() * 100, (norm_pixels < 0.2).mean() * 100
    ],
    'Pneumonie': [
        pneu_pixels.mean(), pneu_pixels.std(),
        (pneu_pixels > 0.5).mean() * 100, (pneu_pixels < 0.2).mean() * 100
    ]
})

display(stats_df)

,Métrique,Normal,Pneumonie
0,Intensité moyenne,0.548625,0.580016
1,Écart-type moyen,0.183590,0.161940
2,Pixels > 0.5 (%),62.339773,70.441420
3,Pixels < 0.2 (%),3.575001,1.478007


### 2.7 Synthèse EDA — Partie II

L'analyse des images médicales pulmonaires met en évidence de subtiles différences structurelles. Visuellement, l'image moyenne de la classe "Pneumonie" présente une opacité bilatérale accrue (aspect plus blanc dans les zones pulmonaires inférieures), typique des infiltrats infectieux par rapport aux poumons sains plus radiotransparents (sombres). La heatmap de corrélation pixel-label corrobore cette observation, indiquant que les régions pulmonaires périphériques et inférieures sont fortement corrélées au diagnostic positif. La projection PCA, n'expliquant qu'une part limitée de la variance avec 2 composantes, montre un fort chevauchement des classes, confirmant qu'une méthode linéaire ou qu'un simple MLP aplatissant l'image serait sous-optimal. Ces observations spatiales justifient sans ambiguïté le déploiement d'un Convolutional Neural Network (CNN) capable de capter ces motifs opacifiants locaux et invariants en translation.

## Section 3 — EDA : Medical Abstracts

### 3.1 Chargement des données

In [18]:
url_train = "https://raw.githubusercontent.com/sebischair/Medical-Abstracts-TC-Corpus/main/medical_tc_train.csv"
url_test  = "https://raw.githubusercontent.com/sebischair/Medical-Abstracts-TC-Corpus/main/medical_tc_test.csv"
df_train = pd.read_csv(url_train)
df_test  = pd.read_csv(url_test)

label_names = {1: 'Digestif', 2: 'Cardiovasculaire', 3: 'Neurologique', 4: 'Oncologique', 5: 'Orthopédique'}
df_train['Classe_Nom'] = df_train['condition_label'].map(label_names)
df_test['Classe_Nom'] = df_test['condition_label'].map(label_names)

print("Shape Train:", df_train.shape)
print("Shape Test:", df_test.shape)
display(df_train.head())

Shape Train: (11550, 3)
Shape Test: (2888, 3)


,condition_label,medical_abstract,Classe_Nom
0,5,Tissue changes around loose prostheses. A cani...,Orthopédique
1,1,Neuropeptide Y and neuron-specific enolase lev...,Digestif
2,2,"Sexually transmitted diseases of the colon, re...",Cardiovasculaire
3,1,Lipolytic factors associated with murine and h...,Digestif
4,3,Does carotid restenosis predict an increased r...,Neurologique


### 3.2 Distribution des classes

In [19]:
plt.figure(figsize=(10, 5))
sns.countplot(y='Classe_Nom', data=df_train, order=df_train['Classe_Nom'].value_counts().index, palette='Set2')
plt.title("Distribution des classes (Train)")
plt.savefig('med_dist_classes.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(6, 6))
colors_pie = sns.color_palette('Set2', n_colors=len(df_train['Classe_Nom'].unique()))
df_train['Classe_Nom'].value_counts().plot.pie(autopct='%1.1f%%', colors=colors_pie)
plt.title("Pourcentage par classe médicale")
plt.ylabel('')
plt.savefig('med_pie_classes.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.3 Analyse de la longueur des textes

In [20]:
df_train['text_length'] = df_train['medical_abstract'].str.split().str.len()

plt.figure(figsize=(12, 6))
sns.histplot(data=df_train, x='text_length', hue='Classe_Nom', kde=True, bins=50, palette='Set2', element='step')
plt.title("Longueur des résumés (en mots) par classe")
plt.savefig('med_text_len_hist.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_train, y='Classe_Nom', x='text_length', palette='Set2')
plt.title("Boxplot de la longueur des textes par classe")
plt.savefig('med_text_len_box.png', dpi=300, bbox_inches='tight')
plt.show()

stat_len = df_train.groupby('Classe_Nom')['text_length'].agg(['min', 'max', 'mean', 'median', 'std']).round(2)
display(stat_len)

,min,max,mean,median,std
Classe_Nom,,,,,
Cardiovasculaire,30,417,178.66,179.0,71.53
Digestif,24,422,174.17,171.0,75.10
Neurologique,31,596,163.34,158.0,70.62
Oncologique,28,578,200.13,205.0,80.26
Orthopédique,24,528,177.06,171.0,75.75


### 3.4 Analyse du vocabulaire

In [21]:
from collections import Counter
from wordcloud import WordCloud

# Basic cleanup
df_train['clean_text'] = df_train['medical_abstract'].str.lower().str.replace('[^\w\s]', '', regex=True)

all_words = ' '.join(df_train['clean_text']).split()
word_counts = Counter(all_words)
print("Taille du vocabulaire total :", len(word_counts))

# Stopwords très basique
stopwords = set(['the', 'of', 'and', 'in', 'to', 'a', 'with', 'patients', 'was', 'for', 'is', 'were', 'that', 'by', 'on', 'as', 'an', 'at', 'from', 'we'])
filtered_words = [w for w in all_words if w not in stopwords and len(w) > 2]
filtered_counts = Counter(filtered_words)

top20 = filtered_counts.most_common(20)
plt.figure(figsize=(10, 6))
sns.barplot(y=[w[0] for w in top20], x=[w[1] for w in top20], palette='viridis')
plt.title("Top 20 mots les plus fréquents (global)")
plt.savefig('med_top20_global.png', dpi=300, bbox_inches='tight')
plt.show()

# Wordclouds par classe
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, (cls_name, group) in enumerate(df_train.groupby('Classe_Nom')):
    text = ' '.join(group['clean_text'])
    wc = WordCloud(width=400, height=300, background_color='white', stopwords=stopwords).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(cls_name)
    axes[i].axis('off')
axes[-1].axis('off') # hide last empty subplot
plt.tight_layout()
plt.savefig('med_wordclouds.png', dpi=300, bbox_inches='tight')
plt.show()

Taille du vocabulaire total : 46842


### 3.5 Tableau de corrélation (PRINCIPAL) et TF-IDF

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(max_features=20, stop_words='english', ngram_range=(1,2))
X_tfidf = vectorizer.fit_transform(df_train['clean_text']).toarray()
features = vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(X_tfidf, columns=features)
df_tfidf['Classe_Nom'] = df_train['Classe_Nom']
mean_tfidf = df_tfidf.groupby('Classe_Nom').mean()

plt.figure(figsize=(12, 6))
sns.heatmap(mean_tfidf, cmap='YlOrRd', annot=True, fmt='.3f')
plt.title("Scores TF-IDF moyens par classe médicale")
plt.savefig('med_tfidf_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Cosine similarity entre classes
vectorizer_full = TfidfVectorizer(max_features=1000, stop_words='english')
X_full = vectorizer_full.fit_transform(df_train['clean_text']).toarray()
df_full = pd.DataFrame(X_full)
df_full['Classe_Nom'] = df_train['Classe_Nom']
mean_vectors = df_full.groupby('Classe_Nom').mean()
cos_sim = cosine_similarity(mean_vectors)

plt.figure(figsize=(8, 6))
sns.heatmap(cos_sim, annot=True, cmap='Blues', xticklabels=mean_vectors.index, yticklabels=mean_vectors.index)
plt.title("Similarité cosinus entre classes médicales")
plt.savefig('med_cos_sim.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.6 Séparabilité des classes (PCA / t-SNE)

In [23]:
from sklearn.manifold import TSNE

# Sous-échantillon pour la vitesse
np.random.seed(42)
sample_idx = np.random.choice(df_train.index, 1000, replace=False)
X_sample = X_full[sample_idx]
y_sample = df_train.loc[sample_idx, 'Classe_Nom']

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sample)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_sample)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y_sample, palette='Set1', alpha=0.7, ax=axes[0])
axes[0].set_title("Projection PCA (TF-IDF)")

sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y_sample, palette='Set1', alpha=0.7, ax=axes[1])
axes[1].set_title("Projection t-SNE (TF-IDF)")

plt.tight_layout()
plt.savefig('med_pca_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.8 Synthèse EDA — Partie III

Le dataset Medical Abstracts expose des textes de littérature clinique de taille modérée (moyenne ~200 mots) mais caractérisés par un vocabulaire de spécialité ultra-spécifique. La distribution des classes est moyennement équilibrée, avec une prédominance de la classe "Oncologique". L'analyse TF-IDF et la projection t-SNE révèlent que les classes ont des terminologies distinctes permettant une bonne séparation (par exemple "tumor" pour l'oncologie, "myocardial" pour le cardiovasculaire). Néanmoins, la matrice de similarité cosinus indique de légères porosités entre spécialités systémiques. L'ordre séquentiel des mots et les dépendances à long terme (le contexte du résumé) étant cruciaux pour désambiguïser des termes médicaux complexes, l'utilisation d'un RNN (ou LSTM/GRU) plutôt qu'un sac de mots s'impose pour capturer la syntaxe clinique complète.

In [24]:
# Effectifs réels mesurés (Medical Abstracts + rappel Pneumonia)
print(f"Pima Diabetes: {len(df)} échantillons")
print(f"PneumoniaMNIST — train: {len(train_data)}, val: {len(val_data)}, test: {len(test_data)}, total: {len(train_data)+len(val_data)+len(test_data)}")
print(f"Medical Abstracts — train: {len(df_train)}, test: {len(df_test)}")

Pima Diabetes: 768 échantillons
PneumoniaMNIST — train: 4708, val: 524, test: 624, total: 5856
Medical Abstracts — train: 11550, test: 2888


## Section 4 — Synthèse EDA Globale

### 4.1 Tableau récapitulatif des 3 datasets

| Critère               | Pima Diabetes | PneumoniaMNIST | Medical Abstracts |
|-----------------------|---------------|----------------|-------------------|
| **Type de données**   | Tabulaire (Clinique) | Image (Radiographie) | Texte (Résumés NLP) |
| **Nb échantillons**   | 768           | 4 708 train / 524 val / 624 test (5 856 total) | voir cellule ci-dessous |
| **Nb features/dim**   | 8 features    | 1×28×28 pixels | Séquences variables |
| **Nb classes**        | 2 (Binaire)   | 2 (Binaire)    | 5 (Multiclasse)   |
| **Déséquilibre classes**| Modéré (65/35) | Fort (74/26)   | Léger             |
| **Valeurs manquantes**| Nombreux "zéros" cachés | Aucune | Aucune |
| **Défi principal**    | Valeurs aberrantes, features corrélées | Opacités floues, basse résolution | Vocabulaire pointu, contexte séquentiel |
| **Architecture choisie**| MLP (Dense)   | CNN (Conv2D)   | RNN/LSTM/Seq2Seq  |

### 4.2 Discussion transversale

La diversité des trois datasets justifie fondamentalement l'évolution des architectures en Deep Learning. 

- **Pour Pima Diabetes (Tabulaire) :** Les données ne comportent aucune structure spatiale ou temporelle. Les features (Glucose, Age) sont indépendantes sur le plan de l'ordre d'entrée, bien que corrélées linéairement et non-linéairement avec la cible. Le **Perceptron Multicouche (MLP)** est l'architecture idoine, capable de combiner ces scalaires via des poids denses globaux pour tracer une frontière de décision complexe.
- **Pour PneumoniaMNIST (Images) :** Contrairement au tabulaire, une image possède une forte autocorrélation locale 2D (les pixels voisins forment des textures pulmonaires). Aplatir ces pixels briserait la topologie radiologique. Le **CNN**, par ses noyaux de convolution glissants, exploite la localité spatiale et l'invariance par translation pour extraire l'opacité infectieuse, peu importe où elle se situe dans le poumon.
- **Pour Medical Abstracts (Texte) :** Le langage naturel impose une dimension séquentielle où l'ordre (syntaxe) détermine le sens clinique. Ni le MLP ni le CNN standard ne peuvent modéliser dynamiquement une phrase de longueur variable avec mémorisation du passé. Le **RNN (et le LSTM/GRU)**, grâce à sa boucle récurrente et son état caché mis à jour pas-à-pas, est mathématiquement taillé pour traiter ce flux d'informations séquentielles.